In [6]:
class SimpleGradebook(object):
    def __init__(self):
        self._grades = {}
    
    def add_student(self,name):
        self._grades[name] = {}
        
    def report_grade(self, name, subject, grade):
        by_subject = self._grades[name]
        grade_list = by_subject.setdefault(subject,[])
        grade_list.append(grade)
    
    def average_grade(self, name):
        by_subject = self._grades[name]
        total, count = 0,0
        for grades in by_subject.values():
            total += sum(grades)
            count += len(grades)
        return total /count
    
book = SimpleGradebook()
book.add_student('Issac Newton')
book.report_grade('Issac Newton','수학',90)
book.report_grade('Issac Newton','과학',80)
print(book.average_grade('Issac Newton'))

85.0


In [12]:
import collections
Grade = collections.namedtuple('Grade',('score','weight'))
## namedtuple을 사용하자

class Subject(object):
    def __init__(self):
        self._grades = []
    
    def report_grade(self, score, weight):
        self._grades.append(Grade(score,weight))
        
    def average_grade(self):
        total, total_weight =0,0
        for grade in self._grades:
            total += grade.score * grade.weight
            total_weight += grade.weight
            
        return total/total_weight
    
class Student(object):
    def __init__(self):
        self._subjects ={}
        
    def subject(self,name):
        if name not in self._subjects:
            self._subjects[name] = Subject()
        return self._subjects[name]
    
    def average_grade(self):
        total, count =0,0
        for subject in self._subjects.values():
            total += subject.average_grade()
            count += 1
        return total / count

class GradeBook(object):
    def __init__(self):
        self._students = {}
    
    def student(self, name):
        if name not in self._students:
            self._students[name] = Student()
        return self._students[name]
    
book=GradeBook()
albert = book.student("알버트 아인슈타인")
math = albert.subject('Math')
math.report_grade(80, 0.10)
eng = albert.subject('English')
eng.report_grade(90, 0.20)
print(albert.average_grade())

85.0


In [14]:
names = ['엄엄엄엄','김칠복','엄칠복','너굴너굴이']
names.sort(key=lambda x: len(x))
print(names)

['김칠복', '엄칠복', '엄엄엄엄', '너굴너굴이']


In [16]:
from collections import defaultdict
def log_missing():
    print('Key added')
    return 0

current = {'green':12,'blue':3}
increments = [
    ('red',5),
    ('blue',17),
    ('orange',9)
]
result = defaultdict(log_missing, current)
print('Before:', dict(result))
for key, amount in increments:
    result[key] += amount
print('After:', dict(result))

Before: {'green': 12, 'blue': 3}
Key added
Key added
After: {'green': 12, 'blue': 20, 'red': 5, 'orange': 9}


In [19]:
current = {'green':12,'blue':3}
increments = [
    ('red',5),
    ('blue',17),
    ('orange',9)
]

def increment_with_report(current, increments):
    added_count = 0
    
    
    def missing():
        nonlocal added_count
        added_count +=1
        return 0
    result = defaultdict(missing,current)
    for key, amount in increments:
        result[key] += amount
    
    return result, added_count

result, count = increment_with_report(current, increments)
assert count == 2

In [20]:
from collections import defaultdict

current = {'green':12,'blue':3}
increments = [
    ('red',5),
    ('blue',17),
    ('orange',9)
]

class CountMissing(object):
    def __init__(self):
        self.added =0
        
    def missing(self):
        self.added +=1
        return 0
    
counter = CountMissing()
result = defaultdict(counter.missing, current)

for key, amount in increments:
    result[key] += amount
assert counter.added == 2

In [26]:
from collections import defaultdict



class BetterCounterMissing(object):
    def __init__(self):
        self.added = 0
    def __call__(self):
        self.added +=1
        return 0
    
counter = BetterCounterMissing()
counter()
assert callable(counter)
result = defaultdict(counter,current)
for key, amount in increments:
    result[key] += amount
assert counter.added ==3


In [ ]:
## Read메서드가 있는 입력 데이터 클래스
class InputData(object):
    def read(self):
        raise NotImplementedError
    
## Disk file에서 데이터를 읽어오도록 구현한 InputData의 서브 클래스
class PathInputData(InputData):
    def __init__(self, path):
        super().__init__()
        self.path = path
    def read(self):
        return open(self.path).read()
    
## 작업 추상화 클래스
class Worker(object):
    def __init__(self, input_data):
        self.input_data = input_data
        self.result = None
    
    def map(self):
        raise NotImplementedError
    
    def reduce(self, other):
        raise NotImplementedError
## Worker 구쳇 서브 클래스    
class LineCountWorker(Worker):
    def map(self):
        data = self.input_data.read()
        self.result = data.count('\n')
        
    def reduce(self,other):
        self.result += other.result
        
        
        

In [ ]:
import os
from tempfile import TemporaryDirectory
from threading import Thread

class InputData(object):
    def read(self):
        raise NotImplementedError
    
## Disk file에서 데이터를 읽어오도록 구현한 InputData의 서브 클래스
class PathInputData(InputData):
    def __init__(self, path):
        super().__init__()
        self.path = path
    def read(self):
        return open(self.path).read()
    
## 작업 추상화 클래스
class Worker(object):
    def __init__(self, input_data):
        self.input_data = input_data
        self.result = None
    
    def map(self):
        raise NotImplementedError
    
    def reduce(self, other):
        raise NotImplementedError
## Worker 구체 서브 클래스    
class LineCountWorker(Worker):
    def map(self):
        data = self.input_data.read()
        self.result = data.count('\n')
        
    def reduce(self,other):
        self.result += other.result


def generate_inputs(data_dir):
    for name in os.listdir(data_dir):
        yield PathInputData(os.path.join(data_dir, name))
        
def create_workers(input_list):
    workers =[]
    for input_data in input_list:
        workers.append(LineCountWorker(input_data))
    return workers

def execute(workers):
    threads = [Thread(target=w.map)for w in workers]
    for thread in threads: thread.start()
    for thread in threads: thread.join()
    
    first, rest = workers[0], workers[1:]
    for worker in rest:
        first.reduce(worker)
    return first.result

def mapredcue(data_dir):
    inputs = generate_inputs(data_dir)
    workers = create_workers(inputs)
    return execute(workers)

def write_test_files(tmpdir):
    test_data = {
        "file1.txt": "apple\nbanana\ncherry\n",            # 3줄
        "file2.txt": "dog\ncat\n",                         # 2줄
        "file3.txt": "python\nmapreduce\nthread\ngil\n"    # 4줄 (합계 9줄)
    }
    
    for filename, content in test_data.items():
        filepath = os.path.join(tmpdir, filename)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
            
with TemporaryDirectory() as tmpdir:
    write_test_files(tmpdir)
    result = mapredcue(tmpdir)
    
print('There are',result, 'lines')

There are 9 lines


In [ ]:
import os
from tempfile import TemporaryDirectory
from threading import Thread

    
## 공통인터페이스의 확장 : 새 inputData 인스턴스 생성 하는 범용 클래스 메서드           
class GenericInputData(object):
    def read(self):
        raise NotImplementedError

    @classmethod
    def generate_inputs(cls, config):
        raise NotImplementedError
    
## Disk file에서 데이터를 읽어오도록 구현한 InputData의 서브 클래스
class PathInputData(GenericInputData):
    def __init__(self, path):
        super().__init__()
        self.path = path
    def read(self):
        return open(self.path).read()
    
    ## 클래스 메서드
    @classmethod
    def generate_inputs(cls,config):
        data_dir = config['data_dir']
        for name in os.listdir(data_dir):
            yield cls(os.path.join(data_dir, name))
    
## 작업 추상화 클래스
class GenericWorker(object):
    def __init__(self, input_data):
        self.input_data = input_data
        self.result = None
    
    def map(self):
        raise NotImplementedError
    
    def reduce(self, other):
        raise NotImplementedError
    
    @classmethod
    def create_workers(cls, input_list, config):
        workers =[]
        for input_data in input_list.generate_inputs(config):
            workers.append(cls(input_data))
        return workers
    
## Worker 구체 서브 클래스    
class LineCountWorker(GenericWorker):
    def map(self):
        data = self.input_data.read()
        self.result = data.count('\n')
        
    def reduce(self,other):
        self.result += other.result
        

# def generate_inputs(data_dir):
#     for name in os.listdir(data_dir):
#         yield PathInputData(os.path.join(data_dir, name))
        


def execute(workers):
    threads = [Thread(target=w.map)for w in workers]
    for thread in threads: thread.start()
    for thread in threads: thread.join()
    
    first, rest = workers[0], workers[1:]
    for worker in rest:
        first.reduce(worker)
    return first.result

def mapredcue(worker_class, input_class, config):
    workers = worker_class.create_workers(input_class, config)
    return execute(workers)

def write_test_files(tmpdir):
    test_data = {
        "file1.txt": "apple\nbanana\ncherry\n",            # 3줄
        "file2.txt": "dog\ncat\n",                         # 2줄
        "file3.txt": "python\nmapreduce\nthread\ngil\n"    # 4줄 (합계 9줄)
    }
    
    for filename, content in test_data.items():
        filepath = os.path.join(tmpdir, filename)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
            
with TemporaryDirectory() as tmpdir:
    write_test_files(tmpdir)
    config = {'data_dir': tmpdir}
    result = mapredcue(LineCountWorker, PathInputData, config)
    
print('There are',result, 'lines')

There are 9 lines


In [14]:
from pprint import pprint


class MyBaseClass(object):
    def __init__(self,value):
        self.value = value
    
class TimesFive(MyBaseClass):
    def __init__(self,value):
        super(__class__,self).__init__(value*5)

        
class PlusTwo(MyBaseClass):
    def __init__(self,value):
        super(__class__,self).__init__(value+2)
        
class ThisWay(TimesFive, PlusTwo):
    def __init__(self,value):
        super(ThisWay,self).__init__(value)

foo = ThisWay(5)
print(foo.value)
        
pprint(ThisWay.mro())


27
[<class '__main__.ThisWay'>,
 <class '__main__.TimesFive'>,
 <class '__main__.PlusTwo'>,
 <class '__main__.MyBaseClass'>,
 <class 'object'>]


In [ ]:
class ToDictMixin(object):
    def to_dict(self):
        return self._traverse_dict(self.__dict__)
    
    def _traverse_dict(self, instance_dict):
        output={}
        for key,value in instance_dict.items():
            output[key] = self._traverse(key,value)
        return output

    def _traverse(self, key, value):
        if isinstance(value, ToDictMixin):
            return value.to_dict()
        elif isinstance(value,dict):
            return self._traverse_dict(value)
        elif isinstance(value,list):
            return[self._traverse(key,i ) for i in value]
        elif hasattr(value, '__dict__'):
            return self._traverse_dict(value.__dict__)
        else:
            return value  

class BinaryTree(ToDictMixin):
    def __init__(self,value, left = None, right = None):
        self.value= value
        self.left = left
        self.right = right
        
tree = BinaryTree(10,
                  left = BinaryTree(7, right=BinaryTree(9)),
                  right=BinaryTree(13, left = BinaryTree(11)))
print(tree.to_dict())



{'value': 10, 'left': {'value': 7, 'left': None, 'right': {'value': 9, 'left': None, 'right': None}}, 'right': {'value': 13, 'left': {'value': 11, 'left': None, 'right': None}, 'right': None}}


In [ ]:
class BinaryTreeWithParent

In [6]:
import json

class JsonMixin(object):
    @classmethod
    def from_json(cls,data):
        kwargs=json.loads(data)
        return cls(**kwargs)
    
    def to_json(self):
        return json.dumps(self.to_dict())
    
class DatacentRack(ToDictMixin, JsonMixin):
    def __init__(self, switch =None, nodes = None):
        self.switch = Switch(**switch)
        self.nodes = [Nodes(**kwargs )for kwargs in nodes]
        
class Switch(ToDictMixin, JsonMixin):
    def __init__(self, ports =None, speed = None):
        self.ports = ports
        self.speed = speed

class Nodes(ToDictMixin, JsonMixin):
    def __init__(self, cores =None, ram = None, disk = None):
        self.cores = cores
        self.ram = ram
        self.disk = disk
        
serialized ="""
{
  "switch": {
    "ports": 48,
    "speed": 10000
  },
  "nodes": [
    {
      "cores": 16,
      "ram": 64,
      "disk": 1000
    },
    {
      "cores": 32,
      "ram": 128,
      "disk": 2000
    }
  ]
}"""

deserialized = DatacentRack.from_json(serialized)
roundtrip = deserialized.to_json()
assert json.loads(serialized) == json.loads(roundtrip)
        


In [8]:
class MyObject(object):
    def __init__(self):
        self.public_field = 5
        self.__private_field = 10
        
    def get_private_field(self):
        return self.__private_field
    
foo = MyObject()
assert foo.public_field == 5
assert foo.get_private_field() == 10

In [ ]:
class MyOtherObject(object):
    def __init__(self):
        self.__private_field = 71
    @classmethod
    def get_private_field_of_instance(cls, insatnce):
        return insatnce.__private_field
    
bar = MyOtherObject()
assert MyOtherObject.get_private_field_of_instance(bar) == 71


class MyParentObject(object):
    def __init__(self):
        self.__private_field = 71

class MyChildObject(MyParentObject):
    def get_private_field(self):
        return self.__private_field

baz = MyChildObject()
assert baz._MyParentObject__private_field == 71



    
    

In [14]:
class ApiClass(object):
    def __init__(self):
        self.__value =5
        
    def get(self):
        return self.__value
    
class Child(ApiClass):
    def __init__(self):
        super().__init__()
        self._value ='hello'
a = Child()
print(a.get(), 'and', a._value, 'should be different')

5 and hello should be different


In [ ]:
from collections.abc import Sequence

class FrequencyList(list):
    def __init__(self,members):
        super().__init__(members)
        
    def frequency(self):
        counts ={}
        for item in self:
            counts.setdefault(item,0)
            counts[item]+=1
        return counts
    
foo = FrequencyList(['a','b','a','c','b','a','d'])
print('Length is', len(foo))
foo.pop()
print('After pop:',repr(foo))
print('Frequency:',foo.frequency())


class BinaryNode(object):
    def __init__(self,value, left=None, right = None):
        self.value= value
        self.left = left
        self.right = right
        
class IndexableNode(Sequence, BinaryNode):
    def _search(self, count, index):
        # 1. 왼쪽 서브트리 탐색 (가장 작은 값들)
        found = None
        if self.left:
            found, count = self.left._search(count, index)
        
        if found:
            return found, count
            
        # 2. 현재 노드 검사
        if count == index:
            return self, count + 1
        
        count += 1 # 
        
        # 3. 오른쪽 서브트리 탐색 (큰 값들)
        if self.right:
            found, count = self.right._search(count, index)
            
        return found, count
        
        
        
    def __getitem__(self,index):
        found,_ = self._search(0, index)
        if not found:
            raise IndexError('Index out of range')
        return found.value
    
    def __len__(self):
        _, count = self._search(0,None)
        return count
    

    
tree = IndexableNode(
    10,
    left = IndexableNode(
        5,
        left=IndexableNode(2),
        right=IndexableNode(
            6, right = IndexableNode(7)
        )
    ),
    right = IndexableNode(15, left=IndexableNode(11))
)

print('Index of 7 is ', tree.index(7))
print('Count of 10 is', tree.count(10))

Length is 7
After pop: ['a', 'b', 'a', 'c', 'b', 'a']
Frequency: {'a': 3, 'b': 2, 'c': 1}
Index of 7 is  3
Count of 10 is 1
